---

#  Ensemble Methods 

**Ensemble methods** combine multiple base learners to produce a single, stronger model. The key insight is that a crowd of weak-but-diverse models often outperforms any single strong model.

In this notebook we apply four families of ensemble techniques to the Wine dataset:

1. **Hard Voting** — combine predictions from several different classifiers by majority vote.
2. **Bagging** (Bootstrap Aggregating) — train many copies of the *same* model on random bootstrap samples; average their predictions.
3. **Random Forests** — a bagging ensemble of decision trees with an additional random feature subsetting at each split.
4. **Boosting** — train models *sequentially*, each correcting the errors of the previous one. We compare **AdaBoost** (reweights samples) vs. **Gradient Boosting** (fits residuals).

Along the way we will:
- Explore the dataset and establish a single-tree baseline.
- Visualize decision boundaries, learning curves, and feature importances.
- Produce a final head-to-head comparison of all methods.

---

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

# Base learners
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Ensemble methods
from sklearn.ensemble import (
    VotingClassifier,
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)

sns.set_theme()
plt.rcParams["figure.figsize"] = (10, 7)

---

## 2. Load & Explore the Wine Dataset

The Wine dataset contains **178 samples** described by **13 chemical features**, belonging to one of **3 cultivar classes**.

---

In [ ]:
data = load_wine()

X_df = pd.DataFrame(data.data, columns=data.feature_names)
y_s  = pd.Series(data.target, name="cultivar")

target_names  = list(data.target_names)
class_mapping = {i: name for i, name in enumerate(target_names)}

print("Feature matrix shape:", X_df.shape)
print("Target shape        :", y_s.shape)
print("\nClasses:", target_names)
print("\nClass distribution:")
print(y_s.value_counts().rename(index=class_mapping))

X_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Class distribution
counts = y_s.map(class_mapping).value_counts()
axes[0].bar(counts.index, counts.values, color=["C0", "C1", "C2"])
axes[0].set_title("Class Distribution", fontsize=14)
axes[0].set_xlabel("Cultivar"); axes[0].set_ylabel("Count")

# Correlation heatmap
corr = X_df.corr()
im   = axes[1].imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr.columns)))
axes[1].set_yticks(range(len(corr.columns)))
axes[1].set_xticklabels(corr.columns, rotation=90, fontsize=7)
axes[1].set_yticklabels(corr.columns, fontsize=7)
axes[1].set_title("Feature Correlation Heatmap", fontsize=14)
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

---

## 3. Train / Test Split & Baseline

We set aside **33 %** of the data for testing (stratified). Then we establish a single **Decision Tree** baseline — the weak learner that most ensemble methods improve upon.

---

In [ ]:
X = X_df.to_numpy()
y = y_s.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, stratify=y, random_state=42
)

print("Training set:", X_train.shape)
print("Test set    :", X_test.shape)

In [ ]:
# Single Decision Tree baseline
baseline = DecisionTreeClassifier(max_depth=3, random_state=42)
baseline.fit(X_train, y_train)

baseline_acc = accuracy_score(y_test, baseline.predict(X_test))
print(f"Baseline (Decision Tree, max_depth=3) accuracy: {baseline_acc:.4f}")

---

## 4. Hard Voting Classifier

### Concept

**Hard voting** is the simplest ensemble: train $m$ diverse classifiers independently, then let them *vote* — the class receiving the most votes wins.

$$\hat{y} = \text{mode}\bigl(\hat{y}_1,\; \hat{y}_2,\; \ldots,\; \hat{y}_m\bigr)$$

Diversity is critical. We combine four very different base learners:
- A Decision Tree
- $k$-Nearest Neighbors
- Logistic Regression
- Support Vector Machine (with a scaled input, wrapped in the pipeline below)

> **Note:** KNN and Logistic Regression are sensitive to feature scale, so we scale the features before those estimators. Since `VotingClassifier` trains each estimator on the same raw data, we wrap scale-sensitive learners manually.

---

In [ ]:
from sklearn.pipeline import make_pipeline

# Individual classifiers
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
knn_clf  = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
lr_clf   = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
svm_clf  = make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=42))

# Evaluate each base learner individually
base_learners = [
    ("Decision Tree",      tree_clf),
    ("KNN (k=5)",          knn_clf),
    ("Logistic Regression",lr_clf),
    ("SVM (RBF)",          svm_clf),
]

print(f"{'Model':<25} {'Accuracy':>10}")
print("-" * 37)
for name, clf in base_learners:
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"{name:<25} {acc:>10.4f}")

In [ ]:
# Hard Voting ensemble
voting_clf = VotingClassifier(
    estimators=[
        ("tree", DecisionTreeClassifier(max_depth=3, random_state=42)),
        ("knn",  make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))),
        ("lr",   make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))),
        ("svm",  make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=42))),
    ],
    voting="hard"
)

voting_clf.fit(X_train, y_train)
voting_acc = accuracy_score(y_test, voting_clf.predict(X_test))

print(f"Hard Voting ensemble accuracy: {voting_acc:.4f}")
print(f"Baseline (single tree) accuracy: {baseline_acc:.4f}")
print(f"Improvement: {voting_acc - baseline_acc:+.4f}")

In [ ]:
# Confusion matrix for the voting classifier
y_pred_voting = voting_clf.predict(X_test)

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_voting)
disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Hard Voting — Confusion Matrix", fontsize=14)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_voting, target_names=target_names))

---

## 5. Bagging (Bootstrap Aggregating)

### Concept

**Bagging** reduces variance by training many copies of a *single* base estimator on different **bootstrap samples** (random samples *with replacement*) of the training data.

$$\hat{y} = \frac{1}{B}\sum_{b=1}^{B} h_b(x) \quad \text{(regression)}$$
$$\hat{y} = \text{mode}\bigl(h_1(x), h_2(x), \ldots, h_B(x)\bigr) \quad \text{(classification)}$$

Because each model sees a different subset of the data, the ensemble is more robust than any individual tree.

**Out-of-Bag (OOB) evaluation:** samples not drawn in a given bootstrap (~37%) serve as a natural validation set — no cross-validation needed.

---

In [ ]:
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=200,
    max_samples=0.8,         # each tree sees 80 % of training rows
    bootstrap=True,          # sampling with replacement
    oob_score=True,          # use out-of-bag samples to estimate accuracy
    n_jobs=-1,
    random_state=42
)

bag_clf.fit(X_train, y_train)
bag_acc     = accuracy_score(y_test, bag_clf.predict(X_test))
bag_oob_acc = bag_clf.oob_score_

print(f"Bagging test accuracy : {bag_acc:.4f}")
print(f"Bagging OOB accuracy  : {bag_oob_acc:.4f}")
print(f"Baseline accuracy     : {baseline_acc:.4f}")

In [ ]:
# Effect of number of estimators on test accuracy
n_estimator_range = [1, 5, 10, 20, 50, 100, 150, 200]
bag_accs = []

for n in n_estimator_range:
    clf = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=n, max_samples=0.8,
        bootstrap=True, n_jobs=-1, random_state=42
    )
    clf.fit(X_train, y_train)
    bag_accs.append(accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(n_estimator_range, bag_accs, marker="o", color="steelblue")
plt.axhline(baseline_acc, linestyle="--", color="red", label=f"Single tree baseline ({baseline_acc:.2f})")
plt.xlabel("Number of Estimators", fontsize=13)
plt.ylabel("Test Accuracy", fontsize=13)
plt.title("Bagging: Accuracy vs. Number of Trees", fontsize=14)
plt.xticks(n_estimator_range)
plt.legend(fontsize=11)
plt.show()

In [ ]:
# Confusion matrix
y_pred_bag = bag_clf.predict(X_test)

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_bag)
disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Bagging (200 trees) — Confusion Matrix", fontsize=14)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_bag, target_names=target_names))

---

## 6. Random Forests

### Concept

**Random Forests** extend bagging with one extra trick: at each split in each tree, only a **random subset of $p$ features** is considered (typically $p = \sqrt{\text{total features}}$). This *decorrelates* the trees — even if one feature is very strong, no single tree is allowed to dominate every split.

| Method       | Bootstrap samples | Random feature subset |
|:-------------|:-----------------:|:---------------------:|
| Bagging      | ✓                 | ✗                     |
| Random Forest| ✓                 | ✓                     |

The added randomness usually reduces variance further and improves generalization.

---

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",     # √13 ≈ 3-4 features per split
    oob_score=True,
    n_jobs=-1,
    random_state=42
)

rf_clf.fit(X_train, y_train)
rf_acc     = accuracy_score(y_test, rf_clf.predict(X_test))
rf_oob_acc = rf_clf.oob_score_

print(f"Random Forest test accuracy : {rf_acc:.4f}")
print(f"Random Forest OOB accuracy  : {rf_oob_acc:.4f}")
print(f"Bagging test accuracy       : {bag_acc:.4f}")
print(f"Baseline accuracy           : {baseline_acc:.4f}")

In [ ]:
# Feature importances
importances = rf_clf.feature_importances_
imp_df = pd.DataFrame({"feature": data.feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 6))
plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1], color="forestgreen")
plt.xlabel("Feature Importance (Mean Gini Decrease)", fontsize=13)
plt.title("Random Forest — Feature Importances", fontsize=14)
plt.tight_layout()
plt.show()

print(imp_df.to_string(index=False))

In [ ]:
# Effect of max_features on test accuracy
max_features_options = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
rf_feat_accs = []

for mf in max_features_options:
    clf = RandomForestClassifier(
        n_estimators=200, max_features=mf,
        n_jobs=-1, random_state=42
    )
    clf.fit(X_train, y_train)
    rf_feat_accs.append(accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(max_features_options, rf_feat_accs, marker="o", color="forestgreen")
plt.axvline(x=int(np.sqrt(13)), linestyle="--", color="gray",
            label=f"sqrt(13) ≈ {int(np.sqrt(13))} (default)")
plt.xlabel("max_features (# features per split)", fontsize=13)
plt.ylabel("Test Accuracy", fontsize=13)
plt.title("Random Forest: Accuracy vs. max_features", fontsize=14)
plt.xticks(max_features_options)
plt.legend(fontsize=11)
plt.show()

In [ ]:
# Confusion matrix
y_pred_rf = rf_clf.predict(X_test)

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
disp.plot(ax=ax, cmap="Greens", colorbar=False)
ax.set_title("Random Forest (200 trees) — Confusion Matrix", fontsize=14)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_rf, target_names=target_names))

---

## 7. Boosting

### Concept

Unlike bagging, **boosting** trains base learners *sequentially*. Each new model focuses on the mistakes of the ensemble so far. The final prediction is a *weighted sum* of all models.

We compare two boosting algorithms:

| Algorithm | How it focuses on mistakes |
|:----------|:---------------------------|
| **AdaBoost** | Re-weights misclassified training samples so the next model pays more attention to hard examples |
| **Gradient Boosting** | Fits each new tree to the **residuals** (negative gradient of the loss) of the current ensemble |

---

### 7.1 AdaBoost

**AdaBoost** (Adaptive Boosting) works as follows:

1. Initialize uniform sample weights $w_i = 1/n$.
2. Train a weak learner $h_t$ on the weighted dataset.
3. Compute its weighted error $\varepsilon_t$.
4. Compute the learner weight $\alpha_t = \frac{1}{2}\ln\!\left(\frac{1-\varepsilon_t}{\varepsilon_t}\right)$.
5. Increase weights of misclassified samples; decrease weights of correct ones.
6. Repeat for $T$ rounds. Final prediction: $\hat{y} = \text{sign}\!\left(\sum_t \alpha_t h_t(x)\right)$.

---

In [ ]:
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),  # "stump"
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)

ada_clf.fit(X_train, y_train)
ada_acc = accuracy_score(y_test, ada_clf.predict(X_test))

print(f"AdaBoost test accuracy  : {ada_acc:.4f}")
print(f"Baseline accuracy       : {baseline_acc:.4f}")

In [ ]:
# Staged accuracy: how does the ensemble improve as stumps are added?
staged_ada_accs = [
    accuracy_score(y_test, y_pred)
    for y_pred in ada_clf.staged_predict(X_test)
]

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(staged_ada_accs) + 1), staged_ada_accs,
         color="darkorange", label="AdaBoost test accuracy")
plt.axhline(baseline_acc, linestyle="--", color="red",
            label=f"Single tree baseline ({baseline_acc:.2f})")
plt.xlabel("Number of Estimators", fontsize=13)
plt.ylabel("Test Accuracy", fontsize=13)
plt.title("AdaBoost: Accuracy vs. Number of Stumps", fontsize=14)
plt.legend(fontsize=11)
plt.show()

In [ ]:
# Effect of learning_rate on AdaBoost
lr_range = [0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
ada_lr_accs = []

for lr in lr_range:
    clf = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=200, learning_rate=lr, random_state=42
    )
    clf.fit(X_train, y_train)
    ada_lr_accs.append(accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(lr_range, ada_lr_accs, marker="o", color="darkorange")
plt.xlabel("Learning Rate", fontsize=13)
plt.ylabel("Test Accuracy", fontsize=13)
plt.title("AdaBoost: Accuracy vs. Learning Rate", fontsize=14)
plt.show()

In [ ]:
# Confusion matrix
y_pred_ada = ada_clf.predict(X_test)

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_ada)
disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
disp.plot(ax=ax, cmap="Oranges", colorbar=False)
ax.set_title("AdaBoost — Confusion Matrix", fontsize=14)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_ada, target_names=target_names))

---

### 7.2 Gradient Boosting

**Gradient Boosting** builds an additive model in a forward stage-wise fashion. At each step $t$, it fits a new tree $h_t$ to the *pseudo-residuals* — the negative gradient of the loss function with respect to the current ensemble's predictions:

$$r_{ti} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F = F_{t-1}}$$

The ensemble is updated:
$$F_t(x) = F_{t-1}(x) + \eta \cdot h_t(x)$$

where $\eta$ is the **learning rate** (shrinkage). Unlike AdaBoost, Gradient Boosting works on any differentiable loss function.

---

In [ ]:
gb_clf = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,       # stochastic gradient boosting — samples 80% of data per tree
    random_state=42
)

gb_clf.fit(X_train, y_train)
gb_acc = accuracy_score(y_test, gb_clf.predict(X_test))

print(f"Gradient Boosting test accuracy : {gb_acc:.4f}")
print(f"AdaBoost test accuracy          : {ada_acc:.4f}")
print(f"Baseline accuracy               : {baseline_acc:.4f}")

In [ ]:
# Staged accuracy
staged_gb_accs = [
    accuracy_score(y_test, y_pred)
    for y_pred in gb_clf.staged_predict(X_test)
]

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(staged_gb_accs) + 1), staged_gb_accs,
         color="purple", label="Gradient Boosting test accuracy")
plt.axhline(baseline_acc, linestyle="--", color="red",
            label=f"Single tree baseline ({baseline_acc:.2f})")
plt.xlabel("Number of Estimators", fontsize=13)
plt.ylabel("Test Accuracy", fontsize=13)
plt.title("Gradient Boosting: Accuracy vs. Number of Trees", fontsize=14)
plt.legend(fontsize=11)
plt.show()

In [ ]:
# Effect of learning_rate on Gradient Boosting
gb_lr_accs = []

for lr in lr_range:
    clf = GradientBoostingClassifier(
        n_estimators=200, max_depth=3,
        learning_rate=lr, subsample=0.8, random_state=42
    )
    clf.fit(X_train, y_train)
    gb_lr_accs.append(accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(lr_range, gb_lr_accs, marker="o", color="purple", label="Gradient Boosting")
plt.plot(lr_range, ada_lr_accs, marker="s", color="darkorange", label="AdaBoost")
plt.xlabel("Learning Rate", fontsize=13)
plt.ylabel("Test Accuracy", fontsize=13)
plt.title("AdaBoost vs. Gradient Boosting: Accuracy vs. Learning Rate", fontsize=14)
plt.legend(fontsize=11)
plt.show()

In [ ]:
# Confusion matrix
y_pred_gb = gb_clf.predict(X_test)

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_gb)
disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
disp.plot(ax=ax, cmap="Purples", colorbar=False)
ax.set_title("Gradient Boosting — Confusion Matrix", fontsize=14)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_gb, target_names=target_names))

---

## 8. AdaBoost vs. Gradient Boosting — Head-to-Head

### Staged accuracy on the same axes

We overlay the learning curves of both boosting methods to see how quickly each one converges and whether either overfits.

---

In [ ]:
plt.figure(figsize=(12, 7))
plt.plot(range(1, len(staged_ada_accs) + 1), staged_ada_accs,
         color="darkorange", alpha=0.85, label="AdaBoost")
plt.plot(range(1, len(staged_gb_accs) + 1), staged_gb_accs,
         color="purple", alpha=0.85, label="Gradient Boosting")
plt.axhline(baseline_acc, linestyle="--", color="red",
            label=f"Single tree baseline ({baseline_acc:.2f})")
plt.xlabel("Number of Estimators", fontsize=13)
plt.ylabel("Test Accuracy", fontsize=13)
plt.title("AdaBoost vs. Gradient Boosting — Staged Accuracy", fontsize=15)
plt.legend(fontsize=12)
plt.show()

In [ ]:
# Cross-validated comparison (5-fold)
print("5-fold CV accuracy (mean ± std):\n")
for name, clf in [
    ("AdaBoost",         AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=200, learning_rate=0.5, random_state=42)),
    ("Gradient Boosting", GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1,
        subsample=0.8, random_state=42)),
]:
    scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy", n_jobs=-1)
    print(f"  {name:<22}  {scores.mean():.4f} ± {scores.std():.4f}")

---

## 9. Final Comparison — All Ensemble Methods

We collect the test accuracy of every model trained in this notebook and plot a side-by-side bar chart.

---

In [ ]:
results = {
    "Baseline\n(Decision Tree)" : baseline_acc,
    "Hard Voting"               : voting_acc,
    "Bagging\n(200 trees)"      : bag_acc,
    "Random Forest\n(200 trees)": rf_acc,
    "AdaBoost\n(200 stumps)"    : ada_acc,
    "Gradient Boosting\n(200 trees)": gb_acc,
}

colors = ["lightcoral", "steelblue", "cadetblue", "forestgreen", "darkorange", "purple"]

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.bar(results.keys(), results.values(), color=colors, edgecolor="black", width=0.6)

for bar, acc in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{acc:.4f}",
            ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylim(0, 1.08)
ax.set_ylabel("Test Accuracy", fontsize=13)
ax.set_title("Ensemble Methods — Wine Dataset Test Accuracy", fontsize=15)
ax.axhline(baseline_acc, color="red", linestyle="--", linewidth=1.2,
           label=f"Baseline ({baseline_acc:.4f})")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\nRanking:")
for rank, (name, acc) in enumerate(
        sorted(results.items(), key=lambda x: x[1], reverse=True), 1):
    print(f"  {rank}. {name.replace(chr(10), ' '):<35} {acc:.4f}")

---

## 10. Summary

In this notebook we explored four families of ensemble methods on the Wine dataset.

### Techniques covered

**Hard Voting**
- Combines diverse base classifiers (Tree, KNN, LR, SVM) by majority vote.
- Strengths: easy to implement; diverse learners complement each other.
- Weakness: all models trained independently; no learning from mistakes.

**Bagging**
- Trains many copies of the same learner on bootstrap samples; reduces variance.
- Out-of-Bag scoring provides a free validation estimate.
- Random Forests add feature subsampling per split, decorrelating trees further and typically outperforming plain bagging.

**Random Forests**
- Best of the parallel ensemble methods on this dataset.
- Built-in feature importance reveals which chemical features most separate the three cultivars.
- Robust default: `max_features="sqrt"` rarely needs tuning.

**AdaBoost vs. Gradient Boosting**
- Both are sequential, corrective ensembles of weak learners.
- **AdaBoost** re-weights samples; uses shallow stumps; sensitive to noisy labels.
- **Gradient Boosting** fits residuals; more flexible (any differentiable loss); usually more accurate but slower to train.
- The learning-rate sweep shows Gradient Boosting is less sensitive to this hyperparameter than AdaBoost on this dataset.

### When to use which?

| Method | Best used when |
|:-------|:---------------|
| Hard Voting | You have several strong but different models and want a quick ensemble |
| Bagging | Your base learner has high variance (e.g., deep trees); you want free OOB validation |
| Random Forest | Tabular data with many features; interpretability via importances is valuable |
| AdaBoost | You want a fast, simple boosting approach; data is clean (low noise) |
| Gradient Boosting | Maximum accuracy on tabular data; noise is low; you can tune carefully |

---